# Pipeline Results Analysis: ChEMBL Antimicrobial Datasets

Cross-pathogen analysis of the data cleaning and selection pipeline for 13 completed pathogens  
(mtuberculosis and pfalciparum excluded — still running).

**Three questions addressed:**
1. **Do we lose too much data?** — funnel from raw ChEMBL assays to final selected datasets
2. **Are we not merging enough?** — failure modes in the merging step (script 15)
3. **What could be done on top?** — data-grounded recommendations

In [ ]:
import os
import warnings
import pandas as pd
import numpy as np
import stylia

warnings.filterwarnings('ignore')

nb_dir = os.path.abspath('.')
root = os.path.dirname(nb_dir)
output_dir = os.path.join(root, 'output')
assets_dir = os.path.join(root, 'assets')
os.makedirs(assets_dir, exist_ok=True)

# Format: slide | Style: ersilia — change with stylia.set_format() / stylia.set_style()
stylia.set_format('slide')
stylia.set_style('ersilia')

PATHOGENS = [
    'abaumannii', 'calbicans', 'campylobacter', 'ecoli', 'efaecium',
    'enterobacter', 'hpylori', 'kpneumoniae', 'ngonorrhoeae',
    'paeruginosa', 'saureus', 'smansoni', 'spneumoniae'
]

LABELS = {
    'abaumannii':   'A. baumannii',
    'calbicans':    'C. albicans',
    'campylobacter':'Campylobacter',
    'ecoli':        'E. coli',
    'efaecium':     'E. faecium',
    'enterobacter': 'Enterobacter',
    'hpylori':      'H. pylori',
    'kpneumoniae':  'K. pneumoniae',
    'ngonorrhoeae': 'N. gonorrhoeae',
    'paeruginosa':  'P. aeruginosa',
    'saureus':      'S. aureus',
    'smansoni':     'S. mansoni',
    'spneumoniae':  'S. pneumoniae',
}

In [ ]:
def safe_read(path, **kwargs):
    try:
        return pd.read_csv(path, **kwargs)
    except Exception:
        return pd.DataFrame()

def count_file_rows(path):
    """Count data rows (excluding header) in a CSV."""
    try:
        with open(path) as f:
            return max(0, sum(1 for _ in f) - 1)
    except Exception:
        return 0

def load_pathogen(p):
    base = os.path.join(output_dir, p)
    return {
        'df_07':     safe_read(f'{base}/07_assays_raw.csv'),
        'n_cpds_07': count_file_rows(f'{base}/07_all_smiles.csv'),
        'df_08':     safe_read(f'{base}/08_assays_cleaned.csv'),
        'df_12':     safe_read(f'{base}/12_datasets.csv'),
        'df_13':     safe_read(f'{base}/13_individual_LM.csv'),
        'df_14':     safe_read(f'{base}/14_individual_selected_LM.csv'),
        'df_15':     safe_read(f'{base}/15_merging_analysis.csv'),
        'df_17':     safe_read(f'{base}/17_final_datasets.csv'),
        'df_18':     safe_read(f'{base}/18_assays_master.csv'),
        'df_20':     safe_read(f'{base}/20_general_datasets.csv'),
        'n_cpds_20': count_file_rows(f'{base}/20_all_smiles.csv'),
    }

print('Loading data for all pathogens...')
DATA = {p: load_pathogen(p) for p in PATHOGENS}
print(f'Done. Loaded {len(DATA)} pathogens.')

---
## 1. Data Loss: Pipeline Funnel

**Question: Do we lose too much data?**

Track the number of assays and unique compounds through each pipeline stage:
- **Raw** → ChEMBL assays matching organism (step 07)
- **Cleaned** → assays with valid direction after preprocessing (step 08)
- **Qt/Mx eligible** → assays with quantitative or mixed dataset type (step 12)
- **Ql only** → assays with only qualitative (text-flag) data — excluded from modelling
- **Modelled** → individually modelled with Random Forest (step 13)
- **Ind. selected** → individually selected (AUROC ≥ 0.70, step 14)
- **Sent to merging** → failed individual selection → sent to merging (step 15)
- **Merged OK** → rescued by merging (step 15–16)
- **Final datasets** → in non-redundant final selection (step 17)

In [ ]:
def compute_funnel(p):
    d = DATA[p]
    r = {'pathogen': p, 'label': LABELS[p]}

    r['raw_assays']   = len(d['df_07'])
    r['raw_cpds']     = d['n_cpds_07']
    r['cleaned_assays'] = len(d['df_08'])

    df_12 = d['df_12']
    if not df_12.empty:
        qt_mx_ids = df_12[df_12['dataset_type'].isin(['quantitative', 'mixed'])]['assay_id'].unique()
        ql_ids    = df_12[df_12['dataset_type'] == 'qualitative']['assay_id'].unique()
        ql_only_ids = set(ql_ids) - set(qt_mx_ids)
        r['eligible_qt_mx'] = len(qt_mx_ids)
        r['ql_only']        = len(ql_only_ids)
    else:
        r['eligible_qt_mx'] = r['ql_only'] = 0

    r['modelled'] = d['df_13']['assay_id'].nunique() if not d['df_13'].empty else 0

    r['ind_selected'] = len(d['df_14']) if not d['df_14'].empty else 0

    df_15 = d['df_15']
    if not df_15.empty:
        r['sent_merging'] = df_15['assay_id'].nunique()
        r['merged_ok']    = df_15[
            df_15['failure_reason'] == 'successfully_merged'
        ]['assay_id'].nunique()
    else:
        r['sent_merging'] = r['merged_ok'] = 0

    df_17 = d['df_17']
    if not df_17.empty and 'selected' in df_17.columns:
        sel = df_17[df_17['selected'] == True]
        r['final_datasets'] = len(sel)
        r['final_cpds']     = int(sel['cpds'].sum()) if not sel.empty else 0
        if 'label' in sel.columns and not sel.empty:
            r['final_A'] = int(sel['label'].str.startswith('A').sum())
            r['final_B'] = int(sel['label'].str.startswith('B').sum())
            r['final_M'] = int(sel['label'].str.startswith('M').sum())
        else:
            r['final_A'] = r['final_B'] = r['final_M'] = 0
    else:
        r['final_datasets'] = r['final_cpds'] = 0
        r['final_A'] = r['final_B'] = r['final_M'] = 0

    r['general_cpds'] = d['n_cpds_20']
    r['coverage']     = r['general_cpds'] / r['raw_cpds'] if r['raw_cpds'] > 0 else 0

    return r

funnel = pd.DataFrame([compute_funnel(p) for p in PATHOGENS])

display_cols = [
    'label', 'raw_assays', 'cleaned_assays', 'eligible_qt_mx', 'ql_only',
    'modelled', 'ind_selected', 'sent_merging', 'merged_ok', 'final_datasets',
    'raw_cpds', 'general_cpds', 'coverage'
]
funnel[display_cols].sort_values('raw_assays', ascending=False)

In [ ]:
# Aggregate totals
numeric_cols = ['raw_assays','cleaned_assays','eligible_qt_mx','ql_only',
                'modelled','ind_selected','sent_merging','merged_ok',
                'final_datasets','raw_cpds','general_cpds']
totals = funnel[numeric_cols].sum()
print('=== AGGREGATE TOTALS (13 pathogens) ===')
for col in numeric_cols:
    print(f'  {col:30s}: {int(totals[col]):>10,}')
print()
print(f'  {"Cleaned / Raw (assays)":30s}: {totals["cleaned_assays"]/totals["raw_assays"]:.1%}')
print(f'  {"Qt/Mx eligible / Cleaned":30s}: {totals["eligible_qt_mx"]/totals["cleaned_assays"]:.1%}')
print(f'  {"Modelled / Qt/Mx eligible":30s}: {totals["modelled"]/totals["eligible_qt_mx"]:.1%}')
print(f'  {"Merged OK / Sent to merging":30s}: {totals["merged_ok"]/totals["sent_merging"]:.1%}')
print(f'  {"General model cpds / Raw cpds":30s}: {totals["general_cpds"]/totals["raw_cpds"]:.1%}')
print()
print(f'  Pathogens with 0 final datasets: {int((funnel["final_datasets"]==0).sum())}')
print(f'  Total final datasets:            {int(funnel["final_datasets"].sum())}')

In [ ]:
def plot_final_datasets_by_condition(ax, funnel):
    df = funnel.sort_values('final_datasets', ascending=True)
    labels = df['label'].tolist()
    y = np.arange(len(labels))
    nc = stylia.NamedColors()

    ax.barh(y, df['final_A'],
            color=nc.blue, label='Condition A')
    ax.barh(y, df['final_B'],
            left=df['final_A'],
            color=nc.purple, label='Condition B')
    ax.barh(y, df['final_M'],
            left=df['final_A'] + df['final_B'],
            color=nc.mint, label='Merged (M)')

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.legend(loc='lower right')
    stylia.label(ax, xlabel='Final selected datasets', title='Final datasets per pathogen', abc='A')


def plot_coverage(ax, funnel):
    df = funnel.sort_values('coverage', ascending=True)
    labels = df['label'].tolist()
    y = np.arange(len(labels))

    cm = stylia.FadingColormap('plum')
    cm.fit(df['coverage'].values)
    colors = cm.transform(df['coverage'].values)

    ax.barh(y, df['coverage'] * 100, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    stylia.label(ax, xlabel='% of raw compounds in ORGANISM models',
                 title='Compound coverage (ORGANISM assays)', abc='B')


fig, axs = stylia.create_figure(1, 2)
plot_final_datasets_by_condition(axs.next(), funnel)
plot_coverage(axs.next(), funnel)
stylia.save_figure(os.path.join(assets_dir, 'pipeline_funnel.png'))
print('Saved pipeline_funnel.png')

---
## 2. Rejection Reasons

**Question: Why do assays fail?**

`15_merging_analysis.csv` records every assay that attempted merging and its outcome.
Note: qualitative-only assays (`ql_only` in the funnel) are excluded from merging **entirely** — they do not even appear here.

Failure reason categories:
- **Merged OK** — rescued by merging, AUROC ≥ 0.70
- **Frac. contribution < 5%** — assay contributes < 5% of its group's union compounds (in both pass 1 and 2)
- **No group partner** — single assay in its experimental group, no partner to merge with
- **Too few compounds** — post-merge compound count below threshold (1,000 or 100)
- **Too few positives** — ≤ 50 active compounds after merging
- **Excluded / no cutoff** — PubChem assay, missing target ID, or no expert cutoff defined

In [ ]:
def map_reason(r):
    if r == 'successfully_merged':
        return 'Merged OK'
    elif r == 'insufficient_fractional_contribution':
        return 'Frac. contribution < 5%'
    elif r.startswith('insufficient_compounds'):
        return 'Too few compounds'
    elif r in ('insufficient_compatible_assays', 'group_qualified_pass1', 'group_qualified_pass2'):
        return 'No group partner'
    elif r.startswith('insufficient_positives'):
        return 'Too few positives'
    elif r in ('excluded_pubchem_assay', 'excluded_no_target_chembl_id', 'no_expert_cutoff'):
        return 'Excluded / no cutoff'
    else:
        return 'Other'


all_reasons = []
for p in PATHOGENS:
    df_15 = DATA[p]['df_15'].copy()
    if df_15.empty:
        continue
    df_15['reason_group'] = df_15['failure_reason'].apply(map_reason)
    df_15['pathogen'] = p
    all_reasons.append(df_15[['pathogen', 'assay_id', 'reason_group']])

reasons_df = pd.concat(all_reasons, ignore_index=True)

# Deduplicate: one (assay, reason) per pathogen
reasons_dedup = reasons_df.drop_duplicates(['pathogen', 'assay_id', 'reason_group'])

# Per-pathogen percentage breakdown
reason_counts = (
    reasons_dedup
    .groupby(['pathogen', 'reason_group'])
    .size()
    .unstack(fill_value=0)
)
reason_pct = reason_counts.div(reason_counts.sum(axis=1), axis=0) * 100
print('Rejection reason breakdown per pathogen (% of assays sent to merging):')
print(reason_pct.round(1).to_string())
print()

# Aggregate across all pathogens
agg_reasons = reasons_dedup.groupby('reason_group').size().sort_values(ascending=False)
print('Aggregate (all pathogens):')
print(agg_reasons.to_string())

In [ ]:
REASON_ORDER = [
    'Merged OK', 'Frac. contribution < 5%', 'No group partner',
    'Too few compounds', 'Too few positives', 'Excluded / no cutoff', 'Other'
]

nc = stylia.NamedColors()
REASON_COLORS = {
    'Merged OK':                nc.mint,
    'Frac. contribution < 5%':  nc.purple,
    'No group partner':         nc.orange,
    'Too few compounds':        nc.plum,
    'Too few positives':        nc.yellow,
    'Excluded / no cutoff':     nc.gray,
    'Other':                    nc.pink,
}


def plot_rejection_per_pathogen(ax, reason_pct):
    present = [r for r in REASON_ORDER if r in reason_pct.columns]
    df = reason_pct[present].copy()
    if 'Merged OK' in df.columns:
        df = df.sort_values('Merged OK', ascending=True)

    labels = [LABELS[p] for p in df.index]
    y = np.arange(len(labels))
    left = np.zeros(len(labels))

    for reason in present:
        ax.barh(y, df[reason].values, left=left,
                color=REASON_COLORS[reason], label=reason)
        left = left + df[reason].values

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlim(0, 100)
    ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left')
    stylia.label(ax, xlabel='% of assays sent to merging',
                 title='Merging rejection reasons per pathogen', abc='A')


def plot_aggregate_rejection(ax, reasons_dedup):
    agg = reasons_dedup.groupby('reason_group').size()
    agg = agg.reindex([r for r in REASON_ORDER if r in agg.index])
    agg = agg.sort_values(ascending=True)

    colors = [REASON_COLORS[r] for r in agg.index]
    y = np.arange(len(agg))
    ax.barh(y, agg.values, color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(agg.index)
    stylia.label(ax, xlabel='Unique assays (all pathogens)',
                 title='Aggregate rejection reasons', abc='B')


fig, axs = stylia.create_figure(1, 2)
plot_rejection_per_pathogen(axs.next(), reason_pct)
plot_aggregate_rejection(axs.next(), reasons_dedup)
stylia.save_figure(os.path.join(assets_dir, 'rejection_reasons.png'))
print('Saved rejection_reasons.png')

---
## 3. Merging Effectiveness

**Question: Are we not merging enough?**

Three angles:
1. **AUROC near-miss** — assays with 0.60 ≤ AUROC < 0.70 individually. These failed the 0.70 threshold, were sent to merging, and may or may not have been rescued.
2. **Lone qualifiers** — assays that had a valid experimental group but no partner to merge with. These represent genuine missed opportunities unrelated to data size.
3. **Campylobacter post-mortem** — why did zero datasets survive for this pathogen?

In [ ]:
# AUROC distribution of individually modelled assays (best AUROC per assay)
auroc_records = []
for p in PATHOGENS:
    df_13 = DATA[p]['df_13']
    if df_13.empty:
        continue
    best = df_13.groupby('assay_id')['avg'].max().reset_index()
    best.columns = ['assay_id', 'auroc']
    best['pathogen'] = p
    auroc_records.append(best)

auroc_all = pd.concat(auroc_records, ignore_index=True)

bins   = [0, 0.60, 0.70, 1.01]
labels = ['< 0.60', 'Near miss\n(0.60–0.70)', 'Selected\n(≥ 0.70)']
auroc_all['outcome'] = pd.cut(auroc_all['auroc'], bins=bins, labels=labels)

print('AUROC outcome across all pathogens (best per assay):')
oc = auroc_all['outcome'].value_counts().sort_index()
for k, v in oc.items():
    print(f'  {str(k):30s}: {v:>6,}  ({v/len(auroc_all):.1%})')
print()
print('Near-miss assays per pathogen:')
nm = auroc_all[auroc_all['outcome'] == 'Near miss\n(0.60–0.70)']
print(nm.groupby('pathogen').size().rename(LABELS).sort_values(ascending=False))

In [ ]:
# Lone qualifier analysis: assays with no group partner
LONE_REASONS = {'group_qualified_pass1', 'group_qualified_pass2', 'insufficient_compatible_assays'}

lone_rows = []
for p in PATHOGENS:
    df_15 = DATA[p]['df_15']
    if df_15.empty:
        lone_rows.append({'pathogen': p, 'label': LABELS[p],
                          'lone': 0, 'total': 0, 'lone_pct': 0})
        continue
    total  = df_15['assay_id'].nunique()
    lone   = df_15[df_15['failure_reason'].isin(LONE_REASONS)]['assay_id'].nunique()
    lone_rows.append({'pathogen': p, 'label': LABELS[p],
                      'lone': lone, 'total': total,
                      'lone_pct': lone / total * 100 if total > 0 else 0})

lone_df = pd.DataFrame(lone_rows)
print('Lone qualifier assays (no merge partner) per pathogen:')
print(lone_df[['label','lone','total','lone_pct']]
      .sort_values('lone_pct', ascending=False)
      .to_string(index=False))
print()
print(f"Total lone qualifier assays: {lone_df['lone'].sum():,} "
      f"({lone_df['lone'].sum() / lone_df['total'].sum():.1%} of all merging-attempted)")

In [ ]:
# Qualitative-only assays excluded from merging entirely
print('Qualitative-only assays (excluded from merging pipeline):')
for p in PATHOGENS:
    n = funnel[funnel['pathogen'] == p]['ql_only'].values[0]
    if n > 0:
        print(f'  {LABELS[p]:25s}: {int(n):>5,}')
total_ql = funnel['ql_only'].sum()
total_qt = funnel['eligible_qt_mx'].sum()
print(f'  {"TOTAL":25s}: {int(total_ql):>5,} ({total_ql/(total_ql+total_qt):.1%} of all eligible assays)')

In [ ]:
# Campylobacter post-mortem
print('=== CAMPYLOBACTER POST-MORTEM ===\n')
camp = DATA['campylobacter']

print(f'Raw assays in ChEMBL:     {len(camp["df_07"]):>6,}')
print(f'Unique raw compounds:     {camp["n_cpds_07"]:>6,}')
print(f'Cleaned assays:           {len(camp["df_08"]):>6,}')

df_12_c = camp['df_12']
if not df_12_c.empty:
    print(f'\nDataset type breakdown:')
    print(df_12_c['dataset_type'].value_counts().to_string())
    qt_mx_c = df_12_c[df_12_c['dataset_type'].isin(['quantitative','mixed'])]
    print(f'\nQt/Mx assays: {qt_mx_c["assay_id"].nunique()}')
    print(f'Max cpds_qt in any assay: {qt_mx_c["cpds_qt"].max()}')
    print(f'Mean cpds_qt: {qt_mx_c["cpds_qt"].mean():.1f}')

df_15_c = camp['df_15']
if not df_15_c.empty:
    print(f'\nMerging failure reasons:')
    print(df_15_c['failure_reason'].value_counts().to_string())
    print(f'\nMax group_compounds (after merge): {df_15_c["group_compounds"].max()}')
    print(f'Groups with ≥ 100 compounds:       {(df_15_c["group_compounds"] >= 100).sum()}')
    print(f'Groups with ≥  50 compounds:       {(df_15_c["group_compounds"] >= 50).sum()}')
    print(f'Groups with ≥  20 compounds:       {(df_15_c["group_compounds"] >= 20).sum()}')

df_20_c = camp['df_20']
if not df_20_c.empty:
    mic = df_20_c[df_20_c['activity_type'] == 'MIC']
    print(f'\nGeneral MIC model (all levels):')
    print(mic[['level','cutoff','n_compounds','n_actives','n_inactives','auroc']].to_string(index=False))

In [ ]:
def plot_auroc_outcomes(ax, auroc_all):
    outcome_order = ['< 0.60', 'Near miss\n(0.60–0.70)', 'Selected\n(≥ 0.70)']
    nc_colors = {
        '< 0.60':                  stylia.NamedColors().gray,
        'Near miss\n(0.60–0.70)':  stylia.NamedColors().orange,
        'Selected\n(≥ 0.70)':      stylia.NamedColors().mint,
    }
    counts = (
        auroc_all.groupby(['pathogen', 'outcome'])
        .size()
        .unstack(fill_value=0)
    )
    pct = counts.div(counts.sum(axis=1), axis=0) * 100
    pct = pct.sort_values('Selected\n(≥ 0.70)', ascending=True)

    labels = [LABELS[p] for p in pct.index]
    y = np.arange(len(labels))
    left = np.zeros(len(labels))

    for col in outcome_order:
        if col not in pct.columns:
            continue
        ax.barh(y, pct[col].values, left=left,
                color=nc_colors[col], label=col)
        left = left + pct[col].values

    ax.set_yticks(y)
    ax.set_yticklabels(labels)
    ax.set_xlim(0, 100)
    ax.legend(loc='lower right')
    stylia.label(ax, xlabel='% of individually modelled assays',
                 title='Individual model AUROC outcomes', abc='A')


def plot_lone_qualifiers(ax, lone_df):
    df = lone_df.sort_values('lone_pct', ascending=True)
    y = np.arange(len(df))

    cm = stylia.FadingColormap('crimson')
    cm.fit(df['lone_pct'].values)
    colors = cm.transform(df['lone_pct'].values)

    ax.barh(y, df['lone_pct'], color=colors)
    ax.set_yticks(y)
    ax.set_yticklabels(df['label'])
    stylia.label(ax, xlabel='% of merging-attempted assays with no group partner',
                 title='Lone qualifier assays', abc='B')


fig, axs = stylia.create_figure(1, 2)
plot_auroc_outcomes(axs.next(), auroc_all)
plot_lone_qualifiers(axs.next(), lone_df)
stylia.save_figure(os.path.join(assets_dir, 'merging_effectiveness.png'))
print('Saved merging_effectiveness.png')

---
## 4. Cross-Pathogen Comparison

General model performance (step 20) pools all ORGANISM assays without the selection filter.
This gives the best-case view of what the data can achieve when merged broadly.

In [ ]:
def get_mic_row(p):
    df_20 = DATA[p]['df_20']
    r = {'pathogen': p, 'label': LABELS[p],
         'mic_auroc': np.nan, 'mic_cpds': 0, 'mic_actives': 0, 'mic_level': ''}
    if df_20.empty:
        return r
    for level in ('middle', 'low', 'high'):
        sub = df_20[
            (df_20['activity_type'] == 'MIC') &
            (df_20['unit'] == 'umol.L-1') &
            (df_20['level'] == level)
        ]
        if not sub.empty:
            row = sub.iloc[0]
            r['mic_auroc']   = row.get('auroc', np.nan)
            r['mic_cpds']    = int(row.get('n_compounds', 0))
            r['mic_actives'] = int(row.get('n_actives', 0))
            r['mic_level']   = level
            break
    return r

mic_df = pd.DataFrame([get_mic_row(p) for p in PATHOGENS])
print('General MIC model summary (middle cutoff preferred):')
print(mic_df[['label','mic_level','mic_cpds','mic_actives','mic_auroc']]
      .sort_values('mic_auroc', ascending=False)
      .to_string(index=False))

In [ ]:
def plot_general_mic_auroc(ax, mic_df):
    df = mic_df.dropna(subset=['mic_auroc']).sort_values('mic_auroc', ascending=True)
    y = np.arange(len(df))
    nc = stylia.NamedColors()

    cm = stylia.FadingColormap('orchid')
    cm.fit(df['mic_auroc'].values)
    colors = cm.transform(df['mic_auroc'].values)

    ax.barh(y, df['mic_auroc'], color=colors)
    ax.axvline(x=0.7, color=nc.gray, linestyle='--', alpha=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(df['label'])
    ax.set_xlim(0.5, 1.0)
    stylia.label(ax, xlabel='AUROC', title='General MIC model (middle cutoff)', abc='A')


def plot_mic_compounds(ax, mic_df):
    df = mic_df.dropna(subset=['mic_auroc']).sort_values('mic_cpds', ascending=True)
    y = np.arange(len(df))
    nc = stylia.NamedColors()

    ax.barh(y, df['mic_actives'], color=nc.mint, label='Actives')
    ax.barh(y, df['mic_cpds'] - df['mic_actives'],
            left=df['mic_actives'], color=nc.plum, label='Inactives')
    ax.set_yticks(y)
    ax.set_yticklabels(df['label'])
    ax.set_xscale('log')
    ax.legend()
    stylia.label(ax, xlabel='Compounds (log)', title='General MIC model compound counts', abc='B')


fig, axs = stylia.create_figure(1, 2)
plot_general_mic_auroc(axs.next(), mic_df)
plot_mic_compounds(axs.next(), mic_df)
stylia.save_figure(os.path.join(assets_dir, 'cross_pathogen_comparison.png'))
print('Saved cross_pathogen_comparison.png')

---
## 5. Key Findings and Recommendations

See printed summary below.

In [ ]:
print('=' * 70)
print('KEY FINDINGS SUMMARY')
print('=' * 70)

total_raw    = int(funnel['raw_assays'].sum())
total_clean  = int(funnel['cleaned_assays'].sum())
total_qt     = int(funnel['eligible_qt_mx'].sum())
total_ql     = int(funnel['ql_only'].sum())
total_model  = int(funnel['modelled'].sum())
total_inds   = int(funnel['ind_selected'].sum())
total_merge  = int(funnel['sent_merging'].sum())
total_mok    = int(funnel['merged_ok'].sum())
total_final  = int(funnel['final_datasets'].sum())
total_raw_cpds  = int(funnel['raw_cpds'].sum())
total_gen_cpds  = int(funnel['general_cpds'].sum())

print()
print('--- Q1: DO WE LOSE TOO MUCH DATA? ---')
print(f'  Raw assays in ChEMBL:          {total_raw:>10,}')
print(f'  After cleaning (dir. assigned):{total_clean:>10,}  ({total_clean/total_raw:.0%} retained)')
print(f'  Qt/Mx eligible for modelling:  {total_qt:>10,}  ({total_qt/total_clean:.0%} of cleaned)')
print(f'  Qualitative-only (excluded):   {total_ql:>10,}  ({total_ql/(total_qt+total_ql):.0%} of eligible assays)')
print(f'  Individually modelled:         {total_model:>10,}  ({total_model/total_qt:.0%} of qt/mx eligible)')
print(f'  Individually selected:         {total_inds:>10,}  ({total_inds/total_model:.0%} of modelled)')
print(f'  Sent to merging:               {total_merge:>10,}')
print(f'  Rescued by merging:            {total_mok:>10,}  ({total_mok/total_merge:.0%} of sent)')
print(f'  Final selected datasets:       {total_final:>10,}')
print()
print(f'  Raw unique compounds:          {total_raw_cpds:>10,}')
print(f'  In ORGANISM general models:    {total_gen_cpds:>10,}  ({total_gen_cpds/total_raw_cpds:.0%} compound coverage)')
print()
print(f'  Pathogens with zero final datasets: {int((funnel["final_datasets"]==0).sum())} (Campylobacter)')

print()
print('--- Q2: ARE WE NOT MERGING ENOUGH? ---')
print('  Top rejection reasons (all pathogens, unique assays):')
for reason, count in agg_reasons.sort_values(ascending=False).head(6).items():
    print(f'    {reason:35s}: {count:>6,}  ({count/len(reasons_dedup):.1%})')
print()
total_lone = int(lone_df['lone'].sum())
total_attempted = int(lone_df['total'].sum())
print(f'  Lone qualifiers (no merge partner): {total_lone:,} assays ({total_lone/total_attempted:.1%})')
print(f'  Qual-only assays excluded from merging: {total_ql:,}')

print()
print('--- Q3: WHAT COULD BE DONE ON TOP? ---')
n_nm = len(auroc_all[auroc_all['outcome'] == 'Near miss\n(0.60–0.70)'])
print(f'  Near-miss assays (AUROC 0.60–0.70): {n_nm:,}')
print()
print('  Recommendations:')
print('  1. FRACTIONAL CONTRIBUTION THRESHOLD: The main bottleneck is frac. contribution')
print('     < 5%. Relaxing to 2-3% could rescue additional assay groups, especially for')
print('     pathogens with many small assays (S. aureus, E. coli).')
print()
print('  2. QUALITATIVE-ONLY MERGING: A dedicated merging track for qualitative-only')
print(f'     assays would rescue {total_ql:,} assays currently excluded. These could')
print('     contribute to binary datasets without requiring numeric values.')
print()
print('  3. CAMPYLOBACTER: The root cause is data sparsity — all assays have < 25')
print('     compounds even after merging. Options: (a) manual curation of Campylobacter')
print('     literature assays for addition to the include list, (b) lower the minimum')
print('     compound threshold specifically for sparse pathogens, or (c) accept that')
print('     ChEMBL simply lacks sufficient Campylobacter phenotypic data at this time.')
print()
print('  4. LONE QUALIFIERS: Assays with a valid experimental group but no partner')
print(f'     account for {total_lone/total_attempted:.0%} of merging-attempted assays. Expanding the')
print('     grouping criteria (e.g., relaxing strain matching or allowing cross-strain')
print('     merging) could pair some of these assays.')
print()
print('  5. AUROC THRESHOLD: {n_nm} assays had AUROC 0.60–0.70 individually. A lower'.format(n_nm=n_nm))
print('     threshold (e.g., 0.65) for data-sparse pathogens (H. pylori, E. faecium,')
print('     N. gonorrhoeae) would yield more datasets at the cost of lower predictive')
print('     quality — a deliberate trade-off the expert team should evaluate.')
print()
print('  6. CROSS-ORGANISM MERGING: E. coli, K. pneumoniae, and Enterobacter belong to')
print('     the same family (Enterobacteriaceae). Merging ORGANISM-level MIC assays across')
print('     these three pathogens could build richer datasets at the cost of biological')
print('     specificity. Worth evaluating for the general model.')

In [ ]:
# Final merging stats table for reference
merge_summary = []
for p in PATHOGENS:
    df_15 = DATA[p]['df_15']
    if df_15.empty:
        merge_summary.append({'label': LABELS[p], 'total_sent': 0, 'merged_ok': 0,
                               'frac_contr': 0, 'lone': 0, 'too_few_cpds': 0})
        continue
    total  = df_15['assay_id'].nunique()
    ok     = df_15[df_15['failure_reason'] == 'successfully_merged']['assay_id'].nunique()
    frac   = df_15[df_15['failure_reason'] == 'insufficient_fractional_contribution']['assay_id'].nunique()
    lone   = df_15[df_15['failure_reason'].isin(LONE_REASONS)]['assay_id'].nunique()
    few_c  = df_15[df_15['failure_reason'].str.startswith('insufficient_compounds')]['assay_id'].nunique()
    merge_summary.append({'label': LABELS[p], 'total_sent': total, 'merged_ok': ok,
                           'frac_contr': frac, 'lone': lone, 'too_few_cpds': few_c})

ms = pd.DataFrame(merge_summary)
ms['merge_rate'] = (ms['merged_ok'] / ms['total_sent'].clip(1) * 100).round(1)
print('Merging statistics per pathogen:')
print(ms.to_string(index=False))